## 0. Set up the environment

In [ ]:
# install dependencies
!pip install -q transformers accelerate peft datasets sentencepiece bitsandbytes evaluate
!pip install -q trl
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 44.5 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# get access to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import math
import torch
import random
import shutil

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    default_data_collator
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

import pandas as pd
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import logging

logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

## 1. Default settings

In [ ]:
# default settings
BASE_MODEL = "Qwen/Qwen1.5-0.5B"  # model name

ROOT_DIR = "/content/drive/MyDrive/Colab_Notebooks/Advanced_CL/final"

# datasets
TRAIN_FILE = f"{ROOT_DIR}/train.jsonl"
VAL_FILE = f"{ROOT_DIR}/val.jsonl"
TEST_FILE = f"{ROOT_DIR}/test.jsonl"

# output path
OUTPUT_DIR = f"{ROOT_DIR}/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)  # create output file

ROUNDS = 5  # number of rounds
SEQ_LENGTH = 128
PROMPT_LENGTH = 64
GEN_LENGTH = 64

BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-5   # learning rate

SEED = 42
DEVICE = "cuda"  # use GPU

In [ ]:
# set random seed
random.seed(SEED)
#np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

# define pad token
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 3. Define necessary functions

load & save jsonl files

In [ ]:
# define functions to load & save jsonl files
# define a function to load jsonl files
def load_jsonl(path):
    # initialise data list
    data = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))

    return data   # list of dictionaries


# define a function to save jsonl files
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

tokenizing

In [ ]:
# tokenizing
# define a function to tokenize text
def tokenize_function(data):

    outputs = tokenizer(
        data["text"],
        truncation=True,  # truncate sequence with more than 80 tokens
        padding="max_length", # pad to 80 tokens if less than 80
        max_length=SEQ_LENGTH
    )

    outputs["labels"] = outputs["input_ids"].copy() # restore labels

    return outputs

Calculate perplexity

In [ ]:
# define a function to calculate perplexity
def compute_perplexity(loss):
    try:
        return math.exp(loss) # calculate perplexity from loss

    except OverflowError:
        return float("inf") # if loss is really large, return "inf"

LoRA fine-tune

In [ ]:
# define a function to load the model & convert to LoRA model
def load_lora_model():
    # load the model
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,   # model name
        torch_dtype=torch.float16,
        device_map="auto",
    )

    # define LoRA hyperparameters
    lora_config = LoraConfig(

        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,

        target_modules=[
            "q_proj",
            "v_proj",
        ]
    )

    # convert to LoRA Qwen model
    model = get_peft_model(
        model,
        lora_config
    )

    model.print_trainable_parameters()

    return model

model fine-tuning function

In [ ]:
# define a function to fine-tune the model
def train_one_round(
    train_path,
    dec_algorithm,
    round_id,
):

    print(f"\n========== TRAIN ROUND {round_id} ==========\n")

    # set output path
    round_dir = f"{OUTPUT_DIR}/{dec_algorithm}/round_{round_id}"
    os.makedirs(round_dir, exist_ok=True)

    # -----------------------------------------------------
    # load datasets
    # -----------------------------------------------------

    # load datasets
    train_data = load_jsonl(train_path)
    val_data = load_jsonl(VAL_FILE)
    test_data = load_jsonl(TEST_FILE)

    # convert to HuggingFace Dataset
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)
    test_dataset = Dataset.from_list(test_data)

    # tokenizing
    train_dataset = train_dataset.map(
        tokenize_function,
        batched=True
    )
    val_dataset = val_dataset.map(
        tokenize_function,
        batched=True
    )
    test_dataset = test_dataset.map(
        tokenize_function,
        batched=True
    )

    # -----------------------------------------------------
    # model
    # -----------------------------------------------------

    model = load_lora_model()

    # -----------------------------------------------------
    # training args
    # -----------------------------------------------------

    training_args = TrainingArguments(
        output_dir=round_dir,   # output path
        per_device_train_batch_size=BATCH_SIZE,   # batch size
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy="epoch",  # evaluate after each epoch
        save_strategy="epoch",    # save checkpoint every epoch
        save_total_limit=1,   # save latest checkpoint
        num_train_epochs=EPOCHS,  # epochs
        learning_rate=LR,   # learning rate
        fp16=True,
        logging_steps=10,   # print loss every 10 steps
        report_to="none",
        load_best_model_at_end=True,   # restore best model in the end
    )

    # -----------------------------------------------------
    # trainer
    # -----------------------------------------------------

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=default_data_collator,
    )

    # -----------------------------------------------------
    # fine-tuning
    # -----------------------------------------------------

    trainer.train()

    # -----------------------------------------------------
    # evaluate
    # -----------------------------------------------------

    # evaluate on validation & test sets
    val_metrics = trainer.evaluate(val_dataset)
    test_metrics = trainer.evaluate(test_dataset)

    # compute loss
    val_loss = val_metrics["eval_loss"]
    test_loss = test_metrics["eval_loss"]

    # compute perplexity
    val_ppl = compute_perplexity(val_loss)
    test_ppl = compute_perplexity(test_loss)

    # restore results to a dictionary (metrics)
    metrics = {
        "round": round_id,
        "val_loss": val_loss,
        "test_loss": test_loss,
        "val_perplexity": val_ppl,
        "test_perplexity": test_ppl,
    }

    # -----------------------------------------------------
    # save results (metrics)
    # -----------------------------------------------------

    metrics_path = f"{round_dir}/metrics.json"

    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    print(metrics)

    # -----------------------------------------------------
    # save final LoRA adapter
    # -----------------------------------------------------

    adapter_path = f"{round_dir}/lora_adapter"

    trainer.model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)

    print(f"\nLoRA adapter saved to:\n{adapter_path}")

    return trainer, train_data
    # trainer: model used for generating next 40 tokens
    # train_data: dataset used for generation

model generation function

In [ ]:
def generate_with_strategy(
    trainer,
    train_data,
    round_id,
    strategy="temperature",
    temperature=1.0,
    top_p=0.9,
    num_beams=6,
    num_beam_groups=3,
    diversity_penalty=1.0
):
    """
    Generate next-round training dataset with  using a fine-tuned model.
    - for DBS outputs: using rerank selection
    """

    print(f"\n========== GENERATE ROUND {round_id + 1} DATA ==========\n")
    print(f"Decoding strategy: {strategy}")

    model = trainer.model
    model.eval()
    model_device = next(model.parameters()).device

    # -----------------------------
    # generation config
    # -----------------------------
    generation_kwargs = {
        "max_new_tokens": GEN_LENGTH,
        "min_new_tokens": GEN_LENGTH,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": None
    }

    # -----------------------------
    # strategy selection
    # -----------------------------
    if strategy == "greedy":
        generation_kwargs.update({
            "do_sample": False,
            "num_beams": 1
        })

    elif strategy == "temperature":
        generation_kwargs.update({
            "do_sample": True,
            "temperature": temperature
        })

    elif strategy == "top_p":
        generation_kwargs.update({
            "do_sample": True,
            "top_p": top_p
        })

    elif strategy == "dbs":
        generation_kwargs.update({
            "do_sample": False,
            "num_beams": num_beams,
            "num_beam_groups": num_beam_groups,
            "diversity_penalty": diversity_penalty,
            "trust_remote_code": True,
            # return all the beams for reranking
            "num_return_sequences": num_beams,
        })

    new_data = []

    # =========================================================
    # helper: rerank function
    # =========================================================
    def sequence_score(seq_ids):
        """
        approximate log-likelihood scoring using model loss
        """
        with torch.no_grad():
            input_ids = seq_ids.unsqueeze(0)
            out = model(input_ids=input_ids, labels=input_ids)
            return -out.loss.item()  # higher is better

    # =========================================================
    # generation loop
    # =========================================================
    for idx, item in enumerate(train_data):

        text = item["text"]
        token_ids = tokenizer.encode(text, add_special_tokens=False)
        token_ids = token_ids[:SEQ_LENGTH]

        if len(token_ids) < SEQ_LENGTH:
            token_ids += [tokenizer.pad_token_id] * (SEQ_LENGTH - len(token_ids))

        prompt_ids = token_ids[:PROMPT_LENGTH]
        input_ids = torch.tensor([prompt_ids], device=model_device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                **generation_kwargs
            )

        # =====================================================
        # DBS: multiple candidates -> rerank
        # =====================================================
        if strategy == "dbs":

            candidates = outputs  # (num_beams, seq_len)

            best_seq = None
            best_score = float("-inf")

            for seq in candidates:
                score = sequence_score(seq)
                if score > best_score:
                    best_score = score
                    best_seq = seq

            generated_ids = best_seq

        else:
            # non-DBS: keep original behavior
            generated_ids = outputs[0]
        generated_ids = generated_ids[:SEQ_LENGTH]

        # =====================================================
        # post-processing
        # =====================================================
        if len(generated_ids) < SEQ_LENGTH:
            pad_tensor = torch.full(
                (SEQ_LENGTH - len(generated_ids),),
                tokenizer.pad_token_id,
                device=generated_ids.device
            )
            generated_ids = torch.cat([generated_ids, pad_tensor])

        eval_ids = generated_ids[PROMPT_LENGTH:]

        # convert tensor to normal Python list so it can be saved into jsonl
        eval_ids = eval_ids.detach().cpu().tolist()
        # remove padding tokens if exist
        eval_ids = [
            token_id for token_id in eval_ids
            if token_id != tokenizer.pad_token_id
        ]

        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        new_item = {
            "title": item["title"],
            "text": generated_text,
            "round": round_id + 1,
            "decoding": strategy,
            # Store generated token ids for generation-level evaluation.
            # This avoids re-tokenizing generated text later.
            "generated_token_ids": eval_ids,
            # restore rerank score
            "rerank_score": float(best_score) if strategy == "dbs" else None
        }

        new_data.append(new_item)

    # =========================================================
    # save
    # =========================================================
    output_path = f"{OUTPUT_DIR}/{strategy}/round_{round_id + 1}/train.jsonl"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_jsonl(new_data, output_path)

    print(f"Saved {len(new_data)} generated samples to: {output_path}")

    return output_path, new_data

### Evaluation Metrics
| Metric | What it measures | Input needed | Direction |
|---|---|---|---|
| Perplexity | Model quality on held-out text | model + tokenizer + test set | lower is better |
| Dist-1 / Dist-2 | Lexical diversity of generated outputs | generated_text | higher is more diverse |
| Self-BLEU | Similarity among generated outputs | generated_text grouped by condition | lower is more diverse |
| Severe repetition rate | Degenerate phrase repetition | generated_text | lower is better |

In [ ]:
# =========================
# Generation-Level Evaluation Metrics
# =========================
# This section evaluates generated continuations only.
#
# It does NOT use TEST_FILE.
# It does NOT compute perplexity.
# It does NOT re-tokenize generated text.
#
# Metrics:
# - Dist-1
# - Dist-2
# - Self-BLEU
# - Severe repetition rate
#
# Input:
# - generated_token_ids saved during generation


def get_ngrams(tokens, n):
    """
    Extract n-grams from a list of token ids.
    """
    if len(tokens) < n:
        return []

    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def distinct_n_from_ids(token_sequences, n=1):
    """
    Compute Dist-n from generated token id sequences.

    Dist-n = number of unique n-grams / total number of n-grams.
    Higher Dist-n means higher token-level diversity.
    """

    all_ngrams = []

    for token_ids in token_sequences:
        all_ngrams.extend(get_ngrams(token_ids, n))

    if len(all_ngrams) == 0:
        return 0.0

    return len(set(all_ngrams)) / len(all_ngrams)


def self_bleu_from_ids(token_sequences):
    """
    Compute Self-BLEU from generated token id sequences.

    Higher Self-BLEU means generated outputs are more similar,
    so diversity is lower.
    """

    token_sequences = [
        seq for seq in token_sequences
        if seq is not None and len(seq) > 0
    ]

    if len(token_sequences) < 2:
        return None

    smoothing = SmoothingFunction().method1
    bleu_scores = []

    for i, hypothesis in enumerate(token_sequences):
        references = [
            ref for j, ref in enumerate(token_sequences)
            if j != i
        ]

        score = sentence_bleu(
            references,
            hypothesis,
            smoothing_function=smoothing
        )

        bleu_scores.append(score)

    return sum(bleu_scores) / len(bleu_scores)


def has_severe_repetition_from_ids(
    token_ids,
    min_phrase_len=2,
    repeat_threshold=3,
    max_phrase_len=10
):
    """
    Check whether a generated token sequence contains severe repetition.

    Severe repetition:
    a phrase of at least 2 tokens appears 3 times or more.
    """

    if len(token_ids) < min_phrase_len * repeat_threshold:
        return False

    effective_max_phrase_len = min(
        max_phrase_len,
        len(token_ids) // repeat_threshold
    )

    for phrase_len in range(min_phrase_len, effective_max_phrase_len + 1):
        phrases = get_ngrams(token_ids, phrase_len)
        counts = Counter(phrases)

        if any(count >= repeat_threshold for count in counts.values()):
            return True

    return False


def severe_repetition_rate_from_ids(token_sequences):
    """
    Compute severe repetition rate from generated token id sequences.

    Lower is better.
    """

    if len(token_sequences) == 0:
        return 0.0

    flags = [
        has_severe_repetition_from_ids(seq)
        for seq in token_sequences
    ]

    return sum(flags) / len(flags)


def evaluate_generation_outputs(generated_data):
    """
    Evaluate generated continuations using generated_token_ids.

    Expected fields in generated_data:
    - decoding
    - generated_token_ids

    This function does not use test set, tokenizer, or perplexity.
    """

    results_df = pd.DataFrame(generated_data)

    if "decoding" not in results_df.columns:
        raise KeyError("generated_data must contain 'decoding'.")

    if "generated_token_ids" not in results_df.columns:
        raise KeyError("generated_data must contain 'generated_token_ids'.")

    rows = []

    for decoding_method, group in results_df.groupby("decoding"):
        token_sequences = group["generated_token_ids"].dropna().tolist()

        rows.append({
            "decoding": decoding_method,
            "num_samples": len(token_sequences),
            "dist_1": distinct_n_from_ids(token_sequences, n=1),
            "dist_2": distinct_n_from_ids(token_sequences, n=2),
            "self_bleu": self_bleu_from_ids(token_sequences),
            "severe_repetition_rate": severe_repetition_rate_from_ids(token_sequences),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("decoding")
        .reset_index(drop=True)
    )


print("Generation-level evaluation metric functions are ready.")

Generation-level evaluation metric functions are ready.


clear up the environment

In [ ]:
# define a function to clear up te cache
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

## 4. Run the whole procedure using different decoding algorithms

### Greedy decoding

In [ ]:
current_train_path = TRAIN_FILE
decoding_algorithm = "greedy"

for r_id in range(ROUNDS):
    print(f"\n{'=' * 80}\nStarting Round {r_id}\n{'=' * 80}")

    # Fine-tune the model for the current round
    trainer, current_train_data = train_one_round(
        current_train_path,
        decoding_algorithm,
        r_id,
    )

    # Generate new training data for the next round
    current_train_path, generated_data_for_eval = generate_with_strategy(
        trainer,
        current_train_data,
        r_id,
        strategy=decoding_algorithm
    )

    # Compute generation-level metrics for the newly generated data.
    # This uses generated_token_ids saved during generation.
    metrics_df = evaluate_generation_outputs(generated_data_for_eval)

    # Display and save evaluation metrics
    display(metrics_df)
    metrics_path = f"{OUTPUT_DIR}/{decoding_algorithm}/round_{r_id + 1}/evaluation_metrics.csv"
    os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
    metrics_df.to_csv(metrics_path, index=False)
    print(f"Saved evaluation metrics to: {metrics_path}")

    # Perform cleanup
    del trainer
    cleanup()

print(f"\n{'=' * 80}\nAll {ROUNDS} rounds completed!\n{'=' * 80}")

### Temperature sampling

In [ ]:
current_train_path = TRAIN_FILE
decoding_algorithm = "temperature"

for r_id in range(ROUNDS):
    print(f"\n{'=' * 80}\nStarting Round {r_id}\n{'=' * 80}")

    # Fine-tune the model for the current round
    trainer, current_train_data = train_one_round(
        current_train_path,
        decoding_algorithm,
        r_id,
    )

    # Generate new training data for the next round
    current_train_path, generated_data_for_eval = generate_with_strategy(
        trainer,
        current_train_data,
        r_id,
        strategy=decoding_algorithm
    )

    # Compute generation-level metrics for the newly generated data.
    # This uses generated_token_ids saved during generation.
    metrics_df = evaluate_generation_outputs(generated_data_for_eval)

    # Display and save evaluation metrics
    display(metrics_df)
    metrics_path = f"{OUTPUT_DIR}/{decoding_algorithm}/round_{r_id + 1}/evaluation_metrics.csv"
    os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
    metrics_df.to_csv(metrics_path, index=False)
    print(f"Saved evaluation metrics to: {metrics_path}")

    # Perform cleanup
    del trainer
    cleanup()

print(f"\n{'=' * 80}\nAll {ROUNDS} rounds completed!\n{'=' * 80}")

### Top-p sampling

In [ ]:
current_train_path = TRAIN_FILE
decoding_algorithm = "top_p"

for r_id in range(ROUNDS):
    print(f"\n{'=' * 80}\nStarting Round {r_id}\n{'=' * 80}")

    # Fine-tune the model for the current round
    trainer, current_train_data = train_one_round(
        current_train_path,
        decoding_algorithm,
        r_id,
    )

    # Generate new training data for the next round
    current_train_path, generated_data_for_eval = generate_with_strategy(
        trainer,
        current_train_data,
        r_id,
        strategy=decoding_algorithm
    )

    # Compute generation-level metrics for the newly generated data.
    # This uses generated_token_ids saved during generation.
    metrics_df = evaluate_generation_outputs(generated_data_for_eval)

    # Display and save evaluation metrics
    display(metrics_df)
    metrics_path = f"{OUTPUT_DIR}/{decoding_algorithm}/round_{r_id + 1}/evaluation_metrics.csv"
    os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
    metrics_df.to_csv(metrics_path, index=False)
    print(f"Saved evaluation metrics to: {metrics_path}")

    # Perform cleanup
    del trainer
    cleanup()

print(f"\n{'=' * 80}\nAll {ROUNDS} rounds completed!\n{'=' * 80}")

### DBS

In [ ]:
current_train_path = TRAIN_FILE
decoding_algorithm = "dbs"

for r_id in range(ROUNDS):
    print(f"\n{'=' * 80}\nStarting Round {r_id}\n{'=' * 80}")

    # Fine-tune the model for the current round
    trainer, current_train_data = train_one_round(
        current_train_path,
        decoding_algorithm,
        r_id,
    )

    # Generate new training data for the next round
    current_train_path, generated_data_for_eval = generate_with_strategy(
        trainer,
        current_train_data,
        r_id,
        strategy=decoding_algorithm
    )

    # Compute generation-level metrics for the newly generated data.
    # This uses generated_token_ids saved during generation.
    metrics_df = evaluate_generation_outputs(generated_data_for_eval)

    # Display and save evaluation metrics
    display(metrics_df)
    metrics_path = f"{OUTPUT_DIR}/{decoding_algorithm}/round_{r_id + 1}/evaluation_metrics.csv"
    os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
    metrics_df.to_csv(metrics_path, index=False)
    print(f"Saved evaluation metrics to: {metrics_path}")

    # Perform cleanup
    del trainer
    cleanup()

print(f"\n{'=' * 80}\nAll {ROUNDS} rounds completed!\n{'=' * 80}")


Starting Round 0

========== TRAIN ROUND 0 ==========



Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

trainable params: 786,432 || all params: 620,356,608 || trainable%: 0.1268


Epoch,Training Loss,Validation Loss
1,1.744572,3.404117
2,1.791647,3.409450
3,1.766487,3.408826


{'round': 4, 'val_loss': 3.4041173458099365, 'test_loss': 3.361747980117798, 'val_perplexity': 30.087726941864577, 'test_perplexity': 28.839557819888096}

LoRA adapter saved to:
/content/drive/MyDrive/Colab_Notebooks/Advanced_CL/final/results/dbs/round_4/lora_adapter

========== GENERATE ROUND 5 DATA ==========

Decoding strategy: dbs
Saved 1000 generated samples to: /content/drive/MyDrive/Colab_Notebooks/Advanced_CL/final/results/dbs/round_5/train.jsonl


,decoding,num_samples,dist_1,dist_2,self_bleu,severe_repetition_rate
0,dbs,1000,0.105203,0.429587,0.10306,0.753


Saved evaluation metrics to: /content/drive/MyDrive/Colab_Notebooks/Advanced_CL/final/results/dbs/round_5/evaluation_metrics.csv

All 5 rounds completed!
